# Check: new prediction code vs old prediction code (7-day MrP)

Compares the prediction outputs of two runs of `4.optimized_mrp_pilot_7d.py` that use **the same fit**:

- **old**: `corrected_predictions` (2026-11-08): GQ written with 18 significant digits, read with cmdstanpy's default `gq.stan_variable()`, poststratified with the per-draw loop.
- **new**: `predictions` (2026-11-09, commit `7bf865e`): GQ written with 9 significant digits, chain CSVs parsed in parallel, poststratified with one sparse matrix product.

For every window and every output file it checks that shapes, columns and group labels are identical, then reports the largest absolute difference in the numeric columns.

What each file contains:
- `national_all_draws.csv`, `group_*_all_draws.csv`, `group_*_category_all_draws.csv`: **every posterior draw** (400) of the national / group / state estimates, so these are compared draw by draw.
- `*_predictions.csv`, `*_category_probs.csv`: summaries of those draws.
- `cell_level_predictions.parquet`: per-cell **summaries only** (mean, median, SD, interval bounds), stored as float32 (about 7 significant digits); single cell draws are not saved.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RUN_DIR = Path(r"D:\Dropbox\is_culture_king\is_culture_king_Yuyue\generated_data\mrp_7d\td10_ad08_w1000_s1000")
BELIEF = "GRASS"
OLD_SUBDIR = "corrected_predictions"   # old code
NEW_SUBDIR = "predictions"             # new code

old_root = RUN_DIR / BELIEF / OLD_SUBDIR
new_root = RUN_DIR / BELIEF / NEW_SUBDIR
new_windows = sorted(p.name for p in new_root.glob("*_stan_output"))
old_windows = sorted(p.name for p in old_root.glob("*_stan_output"))
print(f"new: {len(new_windows)} windows, old: {len(old_windows)} windows")
assert new_windows == old_windows, "The two runs do not have the same windows"


new: 52 windows, old: 52 windows


In [2]:
def read_output(path):
    return pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)


def compare_file(old_path, new_path):
    """Largest absolute difference in the numeric columns; raises if structure or labels differ."""
    a, b = read_output(old_path), read_output(new_path)
    if a.shape != b.shape or list(a.columns) != list(b.columns):
        raise AssertionError(f"Shape/columns differ: {old_path.name} {a.shape} vs {b.shape}")
    numeric = a.select_dtypes("number").columns
    labels = [c for c in a.columns if c not in numeric]
    if not a[labels].astype(str).equals(b[labels].astype(str)):
        raise AssertionError(f"Labels differ in {old_path}")
    if len(numeric) == 0:
        return 0.0, str(a.dtypes.iloc[0]), a.shape
    diff = np.nanmax(np.abs(a[numeric].to_numpy(float) - b[numeric].to_numpy(float)))
    return float(diff), str(a[numeric].dtypes.iloc[-1]), a.shape


rows = []
for window in new_windows:
    for new_path in sorted((new_root / window).iterdir()):
        if new_path.suffix not in (".csv", ".parquet"):
            continue
        old_path = old_root / window / new_path.name
        assert old_path.exists(), f"Missing in old run: {old_path}"
        diff, dtype, shape = compare_file(old_path, new_path)
        rows.append({"window": window, "file": new_path.name, "max_abs_diff": diff,
                     "dtype": dtype, "rows": shape[0], "cols": shape[1]})
results = pd.DataFrame(rows)
print(f"Compared {len(results)} files in {results['window'].nunique()} windows; structure and labels identical in all.")


Compared 1716 files in 52 windows; structure and labels identical in all.


## Largest difference per file type (over all windows)

In [3]:
by_file = (results.groupby("file")
           .agg(max_abs_diff=("max_abs_diff", "max"), dtype=("dtype", "first"),
                windows=("window", "nunique"), rows=("rows", "first"), cols=("cols", "first"))
           .sort_values("max_abs_diff"))
by_file["contains"] = np.select(
    [by_file.index.str.contains("all_draws"), by_file.index == "cell_level_predictions.parquet"],
    ["every posterior draw", "per-cell summaries (float32)"], default="summaries of the draws")
pd.set_option("display.float_format", "{:.2e}".format)
by_file

,max_abs_diff,dtype,windows,rows,cols,contains
file,,,,,,
group_education_category_probs.csv,1.62e-11,float64,52,4,14,summaries of the draws
group_sex_category_probs.csv,1.64e-11,float64,52,4,14,summaries of the draws
group_sex_category_all_draws.csv,1.99e-11,float64,52,4,403,every posterior draw
group_income_category_probs.csv,2.45e-11,float64,52,10,14,summaries of the draws
group_education_category_all_draws.csv,2.64e-11,float64,52,4,403,every posterior draw
group_age_category_probs.csv,2.76e-11,float64,52,12,14,summaries of the draws
group_age_category_all_draws.csv,3.48e-11,float64,52,12,403,every posterior draw
group_eth_category_probs.csv,3.49e-11,float64,52,12,14,summaries of the draws
group_income_category_all_draws.csv,3.51e-11,float64,52,10,403,every posterior draw


## Summary

In [4]:
draw_files = by_file[by_file["contains"] == "every posterior draw"]
csv_files = by_file[by_file.index.str.endswith(".csv")]
cell = by_file.loc["cell_level_predictions.parquet"]
print(f"All posterior draws (national, groups, states; {len(draw_files)} files x {draw_files['windows'].max()} windows): "
      f"max abs diff {draw_files['max_abs_diff'].max():.2e}")
print(f"All CSV outputs: max abs diff {csv_files['max_abs_diff'].max():.2e}")
print(f"Cell-level summaries (float32): max abs diff {cell['max_abs_diff']:.2e}")
# float32 spacing at the largest cell-level value: differences of one or two of these steps are storage rounding
largest = (read_output(new_root / new_windows[0] / "cell_level_predictions.parquet")
           .drop(columns="cell_id").select_dtypes("number").abs().max().max())
print(f"float32 spacing near {largest:.2f}: {np.spacing(np.float32(largest)):.2e}")

All posterior draws (national, groups, states; 16 files x 52 windows): max abs diff 1.20e-09
All CSV outputs: max abs diff 1.20e-09
Cell-level summaries (float32): max abs diff 1.19e-07
float32 spacing near 2.00: 1.19e-07
